# Writing constraints as formulas

`mvr_formula.py` turns a string into MVR constraints. It parses the formula and calls constructors in `mvr_constraints.py` and the operators in `mvr_operators.py`. It builds no automaton of its own, so every formula is shorthand for calls that can be done by hand.

```
never A                    the path never visits A
reach B before A[2]        B is hit before the 2nd visit to A
count(B) in [2,4]          between two and four B's
reach C between 3 and 6    C is reached within a window
```

Two ways to use formulas, both returning a **list** with one entry per top-level `and`. Each feeds a `constraints=` argument directly:

| | returns | for |
| --- | --- | --- |
| `build_mvr(hmm, formula)` | `list[BaseMVR]` | `MVR_CHMM(constraints=...)` |
| `build_mvr_functor(formula)` | `list[MVRConstraint]` | `ConstrainedHiddenMarkovModel(constraints=...)` |

The syntax is summarised in [`FORMULA_CHEATSHEET.md`](FORMULA_CHEATSHEET.md). This notebook uses the same three-state HMM and observation sequence as the other MVR notebooks.

In [ ]:
import itertools
import warnings

import numpy as np
import matplotlib.pyplot as plt
import torch

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.constrained_hmm import ConstrainedHiddenMarkovModel
from conin.hidden_markov_model.mvr_formula import build_mvr, build_mvr_functor
from conin.hidden_markov_model.sampling.ffbs_mvr import ffbs_torch_mvr_chmm


HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={
        "A": 0.28,
        "B": 0.40,
        "C": 0.32,
    },
    transition_probs={
        ("A", "A"): 0.34,
        ("A", "B"): 0.05,
        ("A", "C"): 0.61,
        ("B", "A"): 0.48,
        ("B", "B"): 0.06,
        ("B", "C"): 0.46,
        ("C", "A"): 0.43,
        ("C", "B"): 0.18,
        ("C", "C"): 0.39,
    },
    emission_probs={
        ("A", "lo"): 0.20,
        ("A", "mid"): 0.31,
        ("A", "hi"): 0.49,
        ("B", "lo"): 0.54,
        ("B", "mid"): 0.23,
        ("B", "hi"): 0.23,
        ("C", "lo"): 0.09,
        ("C", "mid"): 0.01,
        ("C", "hi"): 0.90,
    },
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)


def run(mvr, path):
    """Evaluate one MVR on a hidden path, by hand."""
    state = mvr.ini[path[0]]

    for h in path[1:]:
        state = mvr.upd[(state, h)]

    return mvr.evl[state]


def holds(mvrs, path):
    """A formula holds iff every constraint it produced holds."""
    return all(run(mvr, path) for mvr in mvrs)


print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

## 1. Syntax

There are two ways to use formulas:

1. `build_mvr`: input an `hmm` and  formula, outputs a list of `mvr`'s comptible with `hmm`. These can be used to define a `MVR_HMM`.
2.  `build_mvr_functor`: input a formula, outputs a list of `MVRFunctor` factories that build the formula's constraint(s). Use for `initialize_chmm` with `ConstrainedHiddenMarkovModel`.

A formula is checked as it is parsed, so an unknown state or a syntax error is reported against the source text with a caret, not as a failure deep inside a constructor.

NOTE: If a `formula` specifies a conjunction of constraints, these are passed as lists rather than a single `mvr`. Use `()` if a single `mvr` is desired, but in general it's more efficient to pass as a list. See [Composition](##3-composition) for more details.

In [ ]:
# Eager: straight to MVR objects.
constraints = build_mvr(hmm, "reach B before A[2]")
print("build_mvr         ->", [type(mvr).__name__ for mvr in constraints])
print("mediation states   =", len(constraints[0].mediation_states))

# Deferred: functors, built once the model is known.
chmm = ConstrainedHiddenMarkovModel(hmm=hmm, constraints=build_mvr_functor("reach B before A[2]"))
chmm.initialize_chmm()
print("\nbuild_mvr_functor ->", [c.name for c in chmm.constraints])
print("built              =", [type(mvr).__name__ for mvr in chmm.chmm.constraints])
print("backend            =", chmm.constraint_type)

# Errors point at the offending column.
for bad in ["reach B before A[2", "never Z"]:
    try:
        build_mvr(hmm, bad)
    except Exception as exc:
        print(f"\n{type(exc).__name__}: {exc}")

Without the formula interface, expressing "never visit A" meant either explicitly defining the `mvr` or manually calling the correct sequence of constructors and operators. The formula interface allows these calls to be abstracted into a structured string.

The `never A` `mvr` is the same as that in previous notebooks. However, instead of explicitly constructing as was previously done, we can instead specify it with the simple formula `"never a"`.

In [ ]:
# The hand-written version, as it appears in the other notebooks.
mediation_states = ["ok", "violated"]

forbid_A = HomMVR(
    hidden_states=HIDDEN_STATES,
    mediation_states=mediation_states,
    ini={h: ("violated" if h == "A" else "ok") for h in HIDDEN_STATES},
    upd={
        (m, h): ("violated" if m == "violated" or h == "A" else "ok")
        for m in mediation_states
        for h in HIDDEN_STATES
    },
    evl={"ok": True, "violated": False},
)

# The formula.
from_formula = build_mvr(hmm, "never A")

paths = list(itertools.product(HIDDEN_STATES, repeat=4))
agree = all(run(forbid_A, list(p)) == holds(from_formula, list(p)) for p in paths)

print(f"same language on all {len(paths)} paths of length 4: {agree}")
print('one hand-written automaton  ->  "never A"')

## 2. A Compound Constraint Example

Consider the constraint:

> *hit B before visiting A twice*

This can be represented by the formula `"reach B before A[2]"`. This is a compound constraint that is a **precedence** between two constraints: reachability and count.

- `reach b` is the constraint that we must reach `b`. 
- `before` is the operator that compares first-satisfaction times
- `A[2]` is the constriant that we must visit `A` twice.
  
Checked below against an independent reference predicate written directly in Python, by enumerating every path.

In [ ]:
formula = "reach B before A[2]"
constraints = build_mvr(hmm, formula)


def reference(path):
    """Hit B strictly before the 2nd A. Written as the definition, not the recursion."""
    first_B = next((t for t, h in enumerate(path) if h == "B"), None)
    visits_to_A = [t for t, h in enumerate(path) if h == "A"]
    second_A = visits_to_A[1] if len(visits_to_A) > 1 else None

    if first_B is None:
        return False

    return second_A is None or first_B < second_A


def path_weight(path):
    """P(hidden path, observations)."""
    repn = hmm.repn
    idx = [hmm.hidden_to_internal[h] for h in path]

    total = np.log(repn.start_vec[idx[0]])
    for t in range(1, len(idx)):
        total += np.log(repn.transition_mat[idx[t - 1]][idx[t]])
    for t, o in enumerate(observed):
        total += np.log(repn.emission_mat[idx[t]][hmm.observed_to_internal[o]])

    return np.exp(total)


satisfied = evidence = 0.0
disagreements = 0

for path in itertools.product(HIDDEN_STATES, repeat=T):
    path = list(path)
    parsed, expected = holds(constraints, path), reference(path)
    disagreements += parsed != expected

    weight = path_weight(path)
    evidence += weight
    satisfied += weight * parsed

print(f'"{formula}"')
print(f"  lowers to        {len(constraints)} constraint, {len(constraints[0].mediation_states)} mediation states")
print(f"  disagrees with the reference on {disagreements} of {3 ** T} paths")
print(f"  P(satisfied | y) = {satisfied / evidence:.4f}")

## 3. Important Composition-Related Behaviors

There are three composition-related behaviors to know:

1. A top-level `and` splits into separate constraints instead of building the product automaton, since this is cheaper for downstream algorithms. An `and` nested under another operator still builds the product.

2. Temporal operators chain like Python comparisons, so `reach A then reach B then reach C` means `(A then B) and (B then C)`. Note that this also splits into a top-level conjunction of two constraints, and a list of length 2 will be returned.

4. **`between a and b` attaches a `time_range`**, the window over which the constraint is enforced. It has higher priority than `and`, so it lands on the nearest expression. Use parenthesise to window a whole conjunction.

In [ ]:
for formula in [
    "never A and reach B",
    "reach A then reach B then reach C",
    "reach B between 2 and 5",
    "never A and reach B between 2 and 5",
    "(never A and reach B) between 2 and 5",
]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        parsed = build_mvr(hmm, formula)

    windows = [mvr.time_range for mvr in parsed]
    print(f"{formula:<40} -> {len(parsed)} constraint(s), windows {windows}")

## 4. An Example Workflow

Here's an example workflow: write the constraint as a formula,attach it to the HMM, and hand
the result to an algorithm. Sampling is the algorithm here, but step 4 with any of the MVR algorithmic suite: Viterbi, marginal MAP, satisfaction time, or EM instead.

1. **Write the formula.** `reach B before A[2]` — *B is visited before the second visit
   to A*.
2. **Call** `build_mvr(hmm, formula)` to return a list of `mvrs`s.
   (The factory form `build_mvr_functor(formula)` integrates with
   `ConstrainedHiddenMarkovModel`).
3. **Attach** the list to the HMM as an `MVR_CHMM`.
4. **Call an algorithm.**

In [ ]:
# 1. Write the constraint.
formula = "reach B before A[2]"

# 2. Create MVRs, one per top-level conjunct.
constraints = build_mvr(hmm, formula)

# 3. Attach them to the HMM.
chmm = MVR_CHMM(hidden_markov_model=hmm, constraints=constraints)

# 4. Call an algorithm.
paths = ffbs_torch_mvr_chmm(
    chmm,
    observed,
    num_samples=20_000,
    generator=torch.Generator().manual_seed(0),
)

print(f'formula      "{formula}"')
print(f"constraints  {len(constraints)}, "
      f"{[len(mvr.mediation_states) for mvr in constraints]} mediation states")
print(f"drew         {len(paths)} feasible paths of length {T}")

print("\nfirst few draws:")
for path in paths[:5]:
    print("   " + " ".join(path))

print(f"\nevery draw satisfies the formula: {all(holds(constraints, p) for p in paths)}")

The draws are the posterior, so counting how often each hidden state is occupied at each
time shows what the constraint did to it — against the same model with `constraints=[]`.

The formula says B must arrive before the second A, so it should pull B *earlier* and push
A *later*; C is named nowhere in it and should barely move.

In [ ]:
def occupancy(sampled):
    """P(hidden = h at time t), estimated from sampled paths."""
    return {
        h: np.array([np.mean([path[t] == h for path in sampled]) for t in range(T)])
        for h in HIDDEN_STATES
    }


unconstrained = ffbs_torch_mvr_chmm(
    MVR_CHMM(hidden_markov_model=hmm, constraints=[]),
    observed,
    num_samples=20_000,
    generator=torch.Generator().manual_seed(0),
)

free, held = occupancy(unconstrained), occupancy(paths)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)

for ax, h in zip(axes, HIDDEN_STATES):
    ax.plot(range(T), free[h], "o--", color="0.6", label="unconstrained")
    ax.plot(range(T), held[h], "o-", color="C0", label=formula)
    ax.set_title(f"P(hidden = {h})")
    ax.set_xlabel("t")
    ax.set_ylim(-0.03, 1.0)

axes[0].set_ylabel("occupancy")
axes[0].legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

for h in HIDDEN_STATES:
    print(f"{h} at t=0: {free[h][0]:.3f} -> {held[h][0]:.3f}")

A is pushed out of the early times, B is pulled into them, and C is left alone.

## Final Remarks

The formula layer only builds constraints; every algorithm in this folder consumes them. Swap `constraints=[...]` in any other notebook for `build_mvr(hmm, "...")` and it works unchanged.

Anything the grammar cannot say is still reachable by calling `mvr_constraints` and `mvr_operators` directly — the formula layer is a convenience over that algebra, never a replacement for it. The full syntax is in [`FORMULA_CHEATSHEET.md`](FORMULA_CHEATSHEET.md).